# 01 Production: Single Dataset Spectral Exploration

実運用で想定する「1つのCSV内に、波長ごとの吸光度、含水率、日付、サンプル名などの測定情報が入っている」形式を対象にした探索 notebook です。`train.csv` / `test.csv` の2ファイル前提は使わず、この notebook 内で `train / valid / test` に分割します。

## 0. 設定とライブラリ

`DATA_PATH` を実データCSVに変更して実行します。列名は候補から自動判定し、波長列は列名に含まれる 900-1700 nm の数値から検出します。

In [ ]:

from __future__ import annotations

from pathlib import Path
import json
import math
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
os.environ.setdefault('MPLCONFIGDIR', str(Path('/private/tmp') / 'matplotlib-cache'))
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter, find_peaks
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, GroupKFold, GroupShuffleSplit, GridSearchCV, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.utils.validation import check_array, check_is_fitted

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')

for font_name in ['Hiragino Sans', 'AppleGothic', 'Apple SD Gothic Neo', 'Yu Gothic', 'Meiryo', 'Noto Sans CJK JP']:
    try:
        plt.rcParams['font.family'] = font_name
        break
    except Exception:
        pass
plt.rcParams['axes.unicode_minus'] = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path('/Users/ogawatomohiro/myproject')
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'production'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 実運用ではここを実データCSVに変更してください。存在しない場合だけサンプルの train.csv で動作確認します。
DATA_PATH = DATA_DIR / 'production_dataset.csv'
SAMPLE_FALLBACK_PATH = DATA_DIR / 'train.csv'
ENCODINGS = ('cp932', 'utf-8-sig', 'utf-8', 'shift_jis')

TARGET_CANDIDATES = ['含水率', 'mc', 'moisture', 'moisture_content', 'water_content', 'target', 'y']
SAMPLE_CANDIDATES = ['サンプル名', 'サンプルID', 'sample name', 'sample_name', 'sample number', 'sample_id', 'id', 'ID']
DATE_CANDIDATES = ['日付', '測定日', 'date', 'measurement_date', 'measured_at']
GROUP_CANDIDATES = ['ロット', 'lot', 'batch', '樹種', 'species', 'species number', 'sample_name']

# 波長列は 900-1700 nm を主対象にします。実データが範囲外まで含む場合は None にしてください。
WAVELENGTH_RANGE_NM = (900, 1700)
VALID_SIZE = 0.20
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_SPLITS = 5
GROUP_SPLIT_COL = None  # 例: '日付' や 'ロット'。None の場合は目的変数の分布を保つ通常split。


def read_csv_with_fallback(path: Path, encodings=ENCODINGS) -> pd.DataFrame:
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def pick_first_existing(columns, candidates, required=False, label='column'):
    normalized = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in normalized:
            return normalized[key]
    if required:
        raise ValueError(f'{label} not found. candidates={candidates}')
    return None


def parse_wavelength_from_col(col):
    text = str(col).strip()
    if text in {'', 'nan'}:
        return None
    # Examples: 900, 900.0, 900nm, wl_900, Abs_900, absorbance_900_nm
    nums = re.findall(r'(?<!\d)(\d{3,5}(?:\.\d+)?)(?!\d)', text)
    if not nums:
        return None
    values = [float(x) for x in nums]
    # Prefer values in the expected nm range when multiple numbers exist.
    lo, hi = WAVELENGTH_RANGE_NM if WAVELENGTH_RANGE_NM is not None else (0, float('inf'))
    in_range = [v for v in values if lo <= v <= hi]
    return in_range[-1] if in_range else values[-1]


def detect_spectral_columns(df: pd.DataFrame, exclude_cols=None, wavelength_range_nm=WAVELENGTH_RANGE_NM):
    exclude = set(exclude_cols or [])
    records = []
    for col in df.columns:
        if col in exclude:
            continue
        wl = parse_wavelength_from_col(col)
        if wl is None:
            continue
        numeric = pd.to_numeric(df[col], errors='coerce')
        non_na_ratio = numeric.notna().mean()
        if non_na_ratio < 0.90:
            continue
        records.append((col, wl))
    if not records:
        raise ValueError('波長列を検出できませんでした。列名に 900, 901nm, Abs_900 などの波長値が含まれているか確認してください。')
    spec_info = pd.DataFrame(records, columns=['column', 'wavelength_nm']).drop_duplicates('column')
    if wavelength_range_nm is not None:
        lo, hi = wavelength_range_nm
        ranged = spec_info.query('@lo <= wavelength_nm <= @hi').copy()
        if len(ranged) >= 10:
            spec_info = ranged
    spec_info = spec_info.sort_values('wavelength_nm').reset_index(drop=True)
    return spec_info['column'].tolist(), spec_info


def make_y_bins(y, max_bins=8):
    y = pd.Series(y).astype(float)
    n_bins = min(max_bins, max(2, len(y) // 10))
    try:
        bins = pd.qcut(y, q=n_bins, duplicates='drop')
        counts = bins.value_counts()
        if len(counts) >= 2 and counts.min() >= 2:
            return bins.astype(str)
    except Exception:
        pass
    return None


def split_single_dataset(df, target_col, group_col=None, valid_size=VALID_SIZE, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    df = df.dropna(subset=[target_col]).reset_index(drop=True).copy()
    if group_col is not None and group_col in df.columns and df[group_col].nunique() >= 3:
        splitter1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_valid_idx, test_idx = next(splitter1.split(df, groups=df[group_col]))
        train_valid = df.iloc[train_valid_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)
        rel_valid = valid_size / (1 - test_size)
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=rel_valid, random_state=random_state)
        train_idx, valid_idx = next(splitter2.split(train_valid, groups=train_valid[group_col]))
        return train_valid.iloc[train_idx].reset_index(drop=True), train_valid.iloc[valid_idx].reset_index(drop=True), test_df

    stratify = make_y_bins(df[target_col])
    train_valid, test_df = train_test_split(df, test_size=test_size, random_state=random_state, shuffle=True, stratify=stratify)
    rel_valid = valid_size / (1 - test_size)
    stratify_tv = make_y_bins(train_valid[target_col])
    train_df, valid_df = train_test_split(train_valid, test_size=rel_valid, random_state=random_state, shuffle=True, stratify=stratify_tv)
    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred):
    return {
        'r2': float(r2_score(y_true, y_pred)),
        'rmse': rmse(y_true, y_pred),
        'mae': float(mean_absolute_error(y_true, y_pred)),
    }


def odd_window(window_length, n_points):
    w = int(window_length)
    if w % 2 == 0:
        w += 1
    max_w = n_points if n_points % 2 == 1 else n_points - 1
    w = max(3, min(w, max_w))
    return w


def load_production_dataset():
    path = DATA_PATH if DATA_PATH.exists() else SAMPLE_FALLBACK_PATH
    if path == SAMPLE_FALLBACK_PATH:
        print(f'NOTE: {DATA_PATH} がないため、動作確認用に {SAMPLE_FALLBACK_PATH} を読み込みます。')
    df = read_csv_with_fallback(path)
    target_col = pick_first_existing(df.columns, TARGET_CANDIDATES, required=True, label='target column')
    sample_col = pick_first_existing(df.columns, SAMPLE_CANDIDATES, required=False, label='sample id column')
    date_col = pick_first_existing(df.columns, DATE_CANDIDATES, required=False, label='date column')
    group_col = GROUP_SPLIT_COL or pick_first_existing(df.columns, GROUP_CANDIDATES, required=False, label='diagnostic group column')
    exclude_cols = [c for c in [target_col, sample_col, date_col, group_col] if c is not None]
    spectral_cols, spec_info = detect_spectral_columns(df, exclude_cols=exclude_cols)
    df[spectral_cols] = df[spectral_cols].apply(pd.to_numeric, errors='coerce')
    if date_col is not None:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    train_df, valid_df, test_df = split_single_dataset(df, target_col=target_col, group_col=GROUP_SPLIT_COL)
    config = {
        'data_path': str(path),
        'target_col': target_col,
        'sample_col': sample_col,
        'date_col': date_col,
        'diagnostic_group_col': group_col,
        'group_split_col': GROUP_SPLIT_COL,
        'n_spectral_cols': len(spectral_cols),
        'wavelength_min_nm': float(spec_info['wavelength_nm'].min()),
        'wavelength_max_nm': float(spec_info['wavelength_nm'].max()),
        'train_shape': train_df.shape,
        'valid_shape': valid_df.shape,
        'test_shape': test_df.shape,
    }
    return df, train_df, valid_df, test_df, spectral_cols, spec_info, config


def get_xy(part_df, spectral_cols, target_col):
    return part_df[spectral_cols].astype(float), part_df[target_col].astype(float)


def sample_ids(df, sample_col):
    if sample_col is not None and sample_col in df.columns:
        return df[sample_col].values
    return np.arange(len(df))


## 1. データ読み込み、列判定、分割

In [ ]:
df, train_df, valid_df, test_df, spectral_cols, spec_info, config = load_production_dataset()
print(json.dumps(config, ensure_ascii=False, indent=2, default=str))

display(df[[c for c in [config['sample_col'], config['date_col'], config['diagnostic_group_col'], config['target_col']] if c is not None]].head())
display(spec_info.head())
display(spec_info.tail())

for name, part in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
    part.to_csv(OUTPUT_DIR / f'{name}_split.csv', index=False, encoding='utf-8-sig')
print(f'split CSV saved to: {OUTPUT_DIR}')

## 2. 欠損と目的変数分布

In [ ]:
target_col = config['target_col']
sample_col = config['sample_col']
date_col = config['date_col']
group_col = config['diagnostic_group_col']

summary = pd.DataFrame({
    'dataset': ['all', 'train', 'valid', 'test'],
    'samples': [len(df), len(train_df), len(valid_df), len(test_df)],
    'target_missing': [df[target_col].isna().sum(), train_df[target_col].isna().sum(), valid_df[target_col].isna().sum(), test_df[target_col].isna().sum()],
    'spectral_missing_cells': [df[spectral_cols].isna().sum().sum(), train_df[spectral_cols].isna().sum().sum(), valid_df[spectral_cols].isna().sum().sum(), test_df[spectral_cols].isna().sum().sum()],
})
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df[target_col].astype(float), bins=40, kde=True, ax=axes[0])
axes[0].set_title('Moisture distribution: all samples')
axes[0].set_xlabel(target_col)
sns.boxplot(data=pd.concat([train_df.assign(split='train'), valid_df.assign(split='valid'), test_df.assign(split='test')]), x='split', y=target_col, ax=axes[1])
axes[1].set_title('Split balance')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_target_distribution.png', dpi=160)

if group_col is not None and group_col in df.columns:
    display(df.groupby(group_col)[target_col].agg(['count', 'mean', 'std', 'min', 'median', 'max']).sort_values('count', ascending=False).head(30))

## 3. スペクトル表示

横軸は検出した波長 nm です。1 nm刻みでなくても、列名から得た波長順に並べて表示します。

In [ ]:
wavelengths = spec_info['wavelength_nm'].to_numpy(float)
X_all = df[spectral_cols].astype(float)

def plot_spectra(wavelengths, X, title, n_samples=160, color_by=None):
    X_df = pd.DataFrame(X).reset_index(drop=True)
    if n_samples is not None and len(X_df) > n_samples:
        idx = np.linspace(0, len(X_df) - 1, n_samples).astype(int)
        X_plot = X_df.iloc[idx]
        color_data = None if color_by is None else pd.Series(color_by).reset_index(drop=True).iloc[idx]
    else:
        X_plot = X_df
        color_data = None if color_by is None else pd.Series(color_by).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(13, 5))
    if color_data is None:
        ax.plot(wavelengths, X_plot.T, color='#4c78a8', alpha=0.18, linewidth=0.8)
    else:
        labels = color_data.astype(str).fillna('NA')
        for label in labels.unique()[:20]:
            rows = labels == label
            ax.plot(wavelengths, X_plot.loc[rows].T, alpha=0.20, linewidth=0.8, label=label)
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.set_title(title)
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Absorbance')
    fig.tight_layout()
    return fig, ax

fig, ax = plot_spectra(wavelengths, X_all, 'Absorbance spectra', color_by=df[group_col] if group_col in df.columns else None)
fig.savefig(FIGURE_DIR / '01_spectra.png', dpi=160)

mean_spectrum = X_all.mean(axis=0)
std_spectrum = X_all.std(axis=0)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(wavelengths, mean_spectrum, label='mean')
ax.fill_between(wavelengths, mean_spectrum - std_spectrum, mean_spectrum + std_spectrum, alpha=0.25, label='±1 std')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorbance')
ax.set_title('Mean spectrum')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_mean_spectrum.png', dpi=160)

## 4. ピーク・相関・微分スペクトル確認

In [ ]:
def make_derivative(X, window_length=21, polyorder=2, deriv=2):
    w = odd_window(window_length, np.asarray(X).shape[1])
    return savgol_filter(np.asarray(X, dtype=float), window_length=w, polyorder=min(polyorder, w - 1), deriv=deriv, axis=1)

representative_idx = (df[target_col].astype(float) - df[target_col].astype(float).median()).abs().idxmin()
representative = X_all.iloc[representative_idx].to_numpy(float)
smoothed = savgol_filter(representative, window_length=odd_window(21, len(representative)), polyorder=2)
peaks, props = find_peaks(smoothed, prominence=np.nanstd(smoothed) * 0.15)
peak_table = pd.DataFrame({'wavelength_nm': wavelengths[peaks], 'absorbance': smoothed[peaks], 'prominence': props.get('prominences', np.repeat(np.nan, len(peaks)))})
display(peak_table.sort_values('prominence', ascending=False).head(20))

X_deriv2 = make_derivative(X_all, window_length=21, deriv=2)
corr_raw = X_all.apply(lambda s: s.corr(df[target_col].astype(float)), axis=0).to_numpy(float)
corr_deriv = pd.DataFrame(X_deriv2).apply(lambda s: s.corr(df[target_col].astype(float)), axis=0).to_numpy(float)
corr_df = pd.DataFrame({'wavelength_nm': wavelengths, 'corr_raw': corr_raw, 'corr_2nd_derivative': corr_deriv})
corr_df.to_csv(OUTPUT_DIR / '01_correlation_spectrum.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(wavelengths, representative, label='raw', alpha=0.7)
axes[0].plot(wavelengths, smoothed, label='smoothed', linewidth=1.5)
axes[0].scatter(wavelengths[peaks], smoothed[peaks], s=22, color='crimson', label='peaks')
axes[0].set_ylabel('Absorbance')
axes[0].legend()
axes[1].plot(wavelengths, corr_raw, label='raw')
axes[1].plot(wavelengths, corr_deriv, label='2nd derivative')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('Correlation with moisture')
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_peaks_and_correlation.png', dpi=160)

## 5. ヒートマップ

In [ ]:
order = df[target_col].astype(float).sort_values().index
heat = X_deriv2[order]
fig, axes = plt.subplots(2, 1, figsize=(13, 7), gridspec_kw={'height_ratios': [1, 2]}, sharex=True)
axes[0].plot(wavelengths, X_deriv2.mean(axis=0), color='black')
axes[0].set_ylabel('Mean 2nd deriv.')
im = axes[1].imshow(heat, aspect='auto', extent=[wavelengths.min(), wavelengths.max(), len(order), 0], cmap='coolwarm')
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('Samples sorted by moisture')
fig.colorbar(im, ax=axes[1], label='2nd derivative')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_derivative_heatmap.png', dpi=160)